# E-DAIC → pipeline smoke test (Colab GPU)

Runs the multimodal depression pipeline on a real E-DAIC session with **real** BERT/Wav2Vec2 on GPU.

> **This is a plumbing / encoder test, not an objectives eval.** E-DAIC has PHQ-8 (no HAM-D) and is English-only, so the HAM-D accuracy target and the ~3-language robustness gate are *not* exercised here.

In [ ]:
!nvidia-smi -L

## 1. Get the code
Run this, then pick **depression-detection.zip** in the file picker.

In [ ]:
# Upload the source zip (recommended). When the file picker appears,
# choose  depression-detection.zip  from your computer.
from google.colab import files
files.upload()
!unzip -q -o depression-detection.zip -d repo

# --- OR clone instead (only after pushing the branch): ---
# !git clone --depth 1 -b new https://github.com/alexagasha/DIPREGEN-AI.git repo

import os, sys
ROOT = 'repo/depression-detection'
os.chdir(ROOT); sys.path.insert(0, '.')
print('cwd:', os.getcwd())

## 2. Install ML deps  (torch is preinstalled on Colab)

In [ ]:
!pip -q install transformers soundfile

## 3. Mount Drive (holds the E-DAIC session)
Put only the session's **`*_AUDIO.wav` + `*_Transcript.csv`** in Drive — you do **not** need the ~700 MB `features/` folder.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')

## 4. Adapt the E-DAIC session → pipeline schema
Point `--session` at the Drive folder that holds the wav + transcript.

In [ ]:
!python data/edaic/adapt_edaic.py \n  --session '/content/drive/MyDrive/edaic' \n  --pid 304 --split test \n  --out data/edaic/sessions
# add  --labels_csv '/content/drive/MyDrive/edaic/labels.csv'  if you have PHQ-8 labels

## 5. Run with real encoders (GPU)
First run downloads the models (~1-2 GB, a minute or two). The encoders auto-load real models when `transformers` is present (here) and fall back to mocks otherwise.

In [ ]:
import os
os.environ['DEP_TEXT_MODEL']  = 'xlm-roberta-base'            # 768-d, multilingual
os.environ['DEP_AUDIO_MODEL'] = 'facebook/wav2vec2-base-960h' # 768-d

from src.pipelines.text_pipeline import BertTextEncoder
from src.pipelines.audio_pipeline import Wav2Vec2AudioEncoder
from src.fusion.model import FusionHead
from src.fusion.run_pipeline import run_participant

text_enc  = BertTextEncoder()
audio_enc = Wav2Vec2AudioEncoder()
model     = FusionHead()   # random untrained weights — predictions NOT meaningful yet

res = run_participant(304, text_enc, audio_enc, model, data_root='data/edaic/sessions')
print(res)

## What success looks like
- **No** `using mock embeddings` line printed (means the real models loaded on GPU).
- A `RESULT: {...}` dict with `phq9_pred` / `hamd_pred` prints.

The numbers are **not** meaningful (untrained head) — a printed prediction confirms real BERT+Wav2Vec2 + real audio/text flow end-to-end. That is the test passing.

For the multilingual Uganda data later: set `DEP_AUDIO_MODEL='facebook/wav2vec2-large-xlsr-53'` and bump `AUDIO_DIM`/`EMBED_DIM` to **1024** (xlsr-large is not 768).